# Train a Model on MNIST with FiftyOne and Torch

## Import Libraries

In [14]:
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.utils.random as four

In [2]:
import torch
import urllib.request

To run this recipe, you’ll need the mnist_training.py script, which contains a simple PyTorch training loop. The following cell will automatically download the file into your working directory so it can be imported directly.

In [3]:
url = "https://cdn.voxel51.com/tutorials_torch_dataset_examples/notebook_simple_training_example/mnist_training.py"
urllib.request.urlretrieve(url, "mnist_training.py")

('mnist_training.py', <http.client.HTTPMessage at 0x7c184b734b30>)

In [4]:
url = "https://cdn.voxel51.com/tutorials_torch_dataset_examples/notebook_the_cache_field_names_argument/utils.py"
urllib.request.urlretrieve(url, "utils.py")

('utils.py', <http.client.HTTPMessage at 0x7c184b89cc50>)

In [5]:

from notebooks import mnist_training

In [6]:
# torch.multiprocessing.set_start_method('forkserver') Doesnt work on windows
torch.multiprocessing.set_forkserver_preload(['torch', 'fiftyone'])

## Basic Training Example on MNIST

In [7]:
mnist = foz.load_zoo_dataset("mnist")
mnist.persistent = True

100%|██████████| 9.91M/9.91M [00:00<00:00, 10.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 303kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.77MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.96MB/s]

   1% |/------------|   659/60000 [100.9ms elapsed, 9.1s remaining, 6.5K samples/s] 

 100% |█████████████| 60000/60000 [9.1s elapsed, 0s remaining, 6.6K samples/s]      
 100% |█████████████| 10000/10000 [1.5s elapsed, 0s remaining, 6.7K samples/s]         
Dataset info written to '/home/spjor/fiftyone/mnist/info.json'


/home/spjor/PycharmProjects/dat255-pose-estimation/.venv/lib/python3.12/site-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


Loading 'mnist' split 'train'
 100% |█████████████| 60000/60000 [13.2s elapsed, 0s remaining, 4.2K samples/s]       
Loading 'mnist' split 'test'
 100% |█████████████| 10000/10000 [2.3s elapsed, 0s remaining, 4.2K samples/s]      
Dataset 'mnist' created


In [8]:
session = fo.launch_app(mnist, auto=False)

Session launched. Run `session.show()` to open the App in a cell output.


Now let’s say that for our training, we want to define some random subset of our trainset to be a validation set. We can easily do this with FiftyOne.

In [9]:
# remove existing 'train' or 'validation' tags if they exist
mnist.untag_samples(['train', 'validation'])

# create a random split, just on the samples not tagged 'test'
not_test = mnist.match_tags('test', bool=False)
four.random_split(not_test, {'train' : 0.9, 'validation' : 0.1})
print(mnist.count_sample_tags())

{'train': 54000, 'test': 10000, 'validation': 6000}


In [10]:
# subset if we want it
samples = []
samples += mnist.match_tags('train').take(1000).values('id')
for tag in ['test', 'validation']:
    samples += mnist.match_tags(tag).values('id')

subset = mnist.select(samples)

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [12]:
import os
current_path = os.getcwd()

path_to_save_weights = current_path + "/mnist_weights"

In [13]:
mnist_training.main(subset, 10, 10, device, path_to_save_weights)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/spjor/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 32.0MB/s]
Average Validation Loss =   0.495667: 100%|██████████| 375/375 [00:02<00:00, 176.83it/s]


New best lost achieved : 0.5001935362815857. Saving model...


Average Validation Loss =   0.449328: 100%|██████████| 375/375 [00:01<00:00, 260.64it/s]


New best lost achieved : 0.44503626227378845. Saving model...


...

New best lost achieved : 0.15489012002944946. Saving model...


Average Validation Loss =   0.155578: 100%|██████████| 375/375 [00:01<00:00, 259.90it/s]


New best lost achieved : 0.14743243157863617. Saving model...


Average Validation Loss =   0.151626: 100%|██████████| 625/625 [00:06<00:00, 89.50it/s]


Final Test Results:
Loss = 0.15609535574913025
              precision    recall  f1-score   support

    0 - zero       0.98      0.97      0.98       980
     1 - one       0.99      0.99      0.99      1135
     2 - two       0.97      0.97      0.97      1032
   3 - three       0.95      0.97      0.96      1010
    4 - four       0.97      0.94      0.95       982
    5 - five       0.87      0.98      0.92       892
     6 - six       0.98      0.92      0.95       958
   7 - seven       0.94      0.96      0.95      1028
   8 - eight       0.95      0.88      0.92       974
    9 - nine       0.94      0.95      0.94      1009

    accuracy                           0.95     10000
   macro avg       0.95      0.95      0.95     10000
weighted avg       0.96      0.95      0.95     10000

